# Lab: um filtro de spam com random forests

Notebook de aula da página: https://insper.github.io/ML/pt/2026.2/classes/random-forest/

Rode as células em ordem. Em cada passo, responda o **Preveja** com a dupla *antes* de rodar. Nas **Simulações**, escreva o experimento na célula vazia; as soluções e a leitura de cada resultado estão na página da aula.

## Passo 1 — carregar e olhar

**Preveja.** Que palavras você espera ver com muito mais frequência no spam do que no e-mail legítimo? Alguma palavra poderia indicar "legítimo" só por causa de quem coletou os dados?

In [ ]:
import numpy as np
import pandas as pd
from sklearn.datasets import fetch_openml

pd.set_option('display.max_columns', 20)
pd.set_option('display.width', 120)
spam = fetch_openml(data_id=44, as_frame=True)       # Spambase (UCI): 4.601 e-mails
X = spam.data                                        # 57 atributos numéricos
y = (spam.target == '1').astype(int)                 # 1 = spam, 0 = legítimo
print(X.shape)
print(y.value_counts())
tokens = ['word_freq_free', 'word_freq_remove', 'char_freq_%24', 'word_freq_george', 'word_freq_hp']
print(X[tokens].gt(0).groupby(y).mean().round(3))    # fração de e-mails que contêm cada token

## Passo 2 — trancar o conjunto de teste

**Preveja.** Com 921 e-mails no teste, quanto vale um único e-mail em pontos de acurácia?

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42)
print(X_train.shape, X_test.shape)
print(f"spam share: train {y_train.mean():.3f} · test {y_test.mean():.3f}")

## Passo 3 — um baseline e os folds que todos compartilham

**Preveja.** Se 39,4% dos e-mails são spam, quanto acerta um modelo que chama tudo de legítimo? E qual é a ROC AUC dele?

In [ ]:
from sklearn.dummy import DummyClassifier
from sklearn.model_selection import StratifiedKFold, cross_validate

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
folds = {}                                           # as 5 notas por fold de cada modelo, para o passo 5


def evaluate(name, model):
    r = cross_validate(model, X_train, y_train, cv=cv, scoring=['accuracy', 'roc_auc'])
    folds[name] = r
    print(f"{name:14s} accuracy {r['test_accuracy'].mean():.3f} ± {r['test_accuracy'].std():.3f} · "
          f"ROC AUC {r['test_roc_auc'].mean():.3f} ± {r['test_roc_auc'].std():.3f}")


evaluate('dummy', DummyClassifier(strategy='most_frequent'))

## Passo 4 — uma árvore, bagging, random forest, Extra-Trees

**Preveja.** Entre uma árvore e os três ensembles, em qual métrica a diferença vai ser maior, acurácia ou ROC AUC? Por quê?

In [ ]:
from sklearn.ensemble import ExtraTreesClassifier, RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier

evaluate('one tree', DecisionTreeClassifier(random_state=0))
evaluate('bagging', RandomForestClassifier(n_estimators=200, max_features=None,   # todos os 57 atributos
                                           random_state=0, n_jobs=-1))
evaluate('random forest', RandomForestClassifier(n_estimators=200, random_state=0, n_jobs=-1))
evaluate('extra-trees', ExtraTreesClassifier(n_estimators=200, random_state=0, n_jobs=-1))

**Simulação: a floresta é melhor que as próprias árvores?** No primeiro fold do `cv`, ajuste uma floresta de 200 árvores e meça a acurácia de validação de cada árvore sozinha: a média, a melhor e a pior. Compare com o voto da floresta e com uma única árvore crescida em todas as linhas do fold. Alguma árvore da floresta chega ao voto?

In [ ]:
# sua simulação aqui

## Passo 5 — diferenças pareadas, fold a fold

**Preveja.** A floresta está 1,3 ponto de acurácia à frente do bagging, com desvios entre folds de cerca de 0,006. Isso basta para declarar vitória? Que comparação resolve a dúvida?

In [ ]:
for a, b in [('random forest', 'one tree'), ('random forest', 'bagging'), ('extra-trees', 'random forest')]:
    for metric in ['accuracy', 'roc_auc']:
        d = folds[a]['test_' + metric] - folds[b]['test_' + metric]
        print(f"{a} - {b:14s} {metric:8s} {d.round(3)}  ->  {d.mean():+.3f} ± {d.std():.3f}")

## Passo 6 — out-of-bag: a mesma estimativa, com um ajuste

**Preveja.** Cada árvore deixa de fora cerca de 37% dos e-mails. Uma estimativa feita só com as árvores que não viram cada e-mail vai ficar perto da validação cruzada, bem acima ou bem abaixo dela?

In [ ]:
from sklearn.metrics import roc_auc_score

rf = RandomForestClassifier(n_estimators=200, oob_score=True, random_state=0, n_jobs=-1)
rf.fit(X_train, y_train)                             # um ajuste, em todo o X_train
p_oob = rf.oob_decision_function_[:, 1]              # cada e-mail avaliado só por árvores que nunca o viram
print(f"OOB: accuracy {rf.oob_score_:.3f} · ROC AUC {roc_auc_score(y_train, p_oob):.3f}   (1 fit)")
print(f"CV:  accuracy {folds['random forest']['test_accuracy'].mean():.3f} · "
      f"ROC AUC {folds['random forest']['test_roc_auc'].mean():.3f}   (5 fits)")
in_bag = np.array([np.isin(np.arange(len(X_train)), s) for s in rf.estimators_samples_])
n_oob = (~in_bag).sum(axis=0)                        # árvores que NÃO viram cada e-mail
print(f"out-of-bag trees per e-mail: mean {n_oob.mean():.1f} of 200 · min {n_oob.min()} · max {n_oob.max()}")

## Passo 7 — quantas árvores?

**Preveja.** Passadas algumas dezenas de árvores, o erro OOB continua caindo, estaciona ou volta a subir?

In [ ]:
import matplotlib.pyplot as plt

growing = RandomForestClassifier(warm_start=True, oob_score=True, random_state=0, n_jobs=-1)
sizes, oob_error = [10, 25, 50, 100, 200, 400], []
for B in sizes:
    growing.set_params(n_estimators=B).fit(X_train, y_train)   # warm_start: só as árvores novas são crescidas
    oob_error.append(1 - growing.oob_score_)
print(pd.Series(oob_error, index=pd.Index(sizes, name='trees'), name='OOB error').round(4).to_string())
plt.plot(sizes, oob_error, 'o-')
plt.xscale('log'); plt.xlabel('number of trees'); plt.ylabel('OOB error')
plt.show()

**Simulação: árvores rasas, quantas forem.** O passo 7 mostrou o erro OOB estacionar com árvores completas. Repita com árvores de profundidade 3 (`max_depth=3`), para 10, 50, 200 e 400 árvores. Acrescentar árvores leva a floresta rasa até o erro da floresta de árvores completas?

In [ ]:
# sua simulação aqui

**Para fechar a parte fundamental.** Em uma frase cada: (1) por que o voto de 200 árvores acerta mais que a melhor delas? (2) Por que o erro OOB é honesto, se toda árvore foi treinada no `X_train`? (3) Em que situação acrescentar árvores não ajuda nada?

## Sua vez (fundamental)

Para casa:

1. **Sem as duplicatas.** Refaça os passos 3 a 6 depois de `X = X.drop_duplicates()` (e do `y` correspondente). Quanto caem a acurácia e a ROC AUC de CV? O erro OOB acompanha a queda?
2. **Amostras bootstrap menores.** Use `max_samples=0.2` (cada árvore vê 20% das linhas) e depois `0.5`; compare a ROC AUC de CV e o tempo de ajuste com o padrão. Quantas árvores out-of-bag cada e-mail recebe agora? A estimativa OOB continua perto da CV?
3. **Uma prévia da aula 22.** Acrescente `evaluate('boosting', HistGradientBoostingClassifier(random_state=0))` (de `sklearn.ensemble`) ao passo 4 e calcule sua diferença pareada com a random forest em ROC AUC, como no passo 5. É vitória, derrota ou empate?

## Passo 8 — ajustar `max_features` e `min_samples_leaf`

**Preveja.** Menos atributos por divisão deixam cada árvore mais fraca. Você espera o melhor `max_features` perto de 1, perto do padrão (7) ou perto de 57? E folhas de 1 ou de 5 e-mails?

In [ ]:
from sklearn.model_selection import GridSearchCV

grid = GridSearchCV(RandomForestClassifier(n_estimators=200, random_state=0, n_jobs=-1),
                    {'max_features': [1, 2, 4, 7, 15], 'min_samples_leaf': [1, 5]},
                    cv=cv, scoring='roc_auc')
grid.fit(X_train, y_train)
res = pd.DataFrame(grid.cv_results_)
print(res.pivot(index='param_min_samples_leaf', columns='param_max_features',
                values='mean_test_score').round(4))
best_std = res.loc[grid.best_index_, 'std_test_score']
print(f"best: {grid.best_params_} · ROC AUC {grid.best_score_:.4f} ± {best_std:.4f}")
ties = res[res['mean_test_score'] >= grid.best_score_ - best_std]
print(f"{len(ties)} of {len(res)} configurations within one std of the best:",
      list(zip(ties['param_max_features'], ties['param_min_samples_leaf'])))

**Simulação: a busca de graça, com o OOB.** A grade do passo 8 fez 50 ajustes (10 configurações × 5 folds). Refaça a linha de `min_samples_leaf=1` com um único ajuste por configuração, lendo a ROC AUC das probabilidades out-of-bag (`oob_decision_function_`). O OOB ordena os valores de `max_features` como a validação cruzada?

In [ ]:
# sua simulação aqui

## Passo 9 — um limiar que protege o e-mail legítimo, escolhido out-of-bag

**Preveja.** Para mandar no máximo 1% dos e-mails legítimos para a pasta de spam, o limiar precisa subir ou descer em relação a 0,5? Onde dá para escolhê-lo sem tocar no teste?

In [ ]:
final = RandomForestClassifier(n_estimators=500, oob_score=True, random_state=0, n_jobs=-1,
                               **grid.best_params_).fit(X_train, y_train)
p_oob = final.oob_decision_function_[:, 1]
legit = (y_train == 0).to_numpy()
print(" threshold  legit flagged (of 2230)  spam caught")
for t in [0.5, 0.6, 0.65, 0.7, 0.8]:
    flag = p_oob >= t
    print(f"{t:10.2f}  {flag[legit].sum():5d} ({flag[legit].mean():.2%})        {flag[~legit].mean():.1%}")
thresholds = np.arange(0.50, 0.96, 0.05)
chosen = next(t for t in thresholds if (p_oob[legit] >= t).mean() <= 0.01)
print(f"chosen threshold: {chosen:.2f}")

## Passo 10 — o único teste

**Preveja.** No limiar escolhido, a fração de e-mails legítimos perdidos no teste vai ficar perto dos 0,85% previstos pelo OOB? E o recall de spam, sobe ou cai em relação ao limiar 0,5?

In [ ]:
from sklearn.metrics import confusion_matrix, precision_score, recall_score

p_test = final.predict_proba(X_test)[:, 1]
print(f"test ROC AUC {roc_auc_score(y_test, p_test):.3f}")
for t in [0.5, chosen]:
    pred = (p_test >= t).astype(int)
    tp, fn, fp, tn = confusion_matrix(y_test, pred, labels=[1, 0]).ravel()
    print(f"threshold {t:.2f}: legit lost {fp} of {tn + fp} · spam missed {fn} of {fn + tp} · "
          f"precision {precision_score(y_test, pred):.3f} · recall {recall_score(y_test, pred):.3f}")

## Passo 11 — para onde a floresta olha?

**Preveja.** Uma coluna de números de série, sem nenhum sinal, vai aparecer bem colocada em alguma das duas importâncias? Em qual?

In [ ]:
from sklearn.inspection import permutation_importance

rng = np.random.default_rng(0)
Xn_train = X_train.assign(random_id=rng.permutation(len(X_train)))   # mais uma coluna de puro ruído
Xn_test = X_test.assign(random_id=rng.permutation(len(X_test)))
probe = RandomForestClassifier(n_estimators=200, random_state=0, n_jobs=-1,
                               **grid.best_params_).fit(Xn_train, y_train)
perm = permutation_importance(probe, Xn_test, y_test, scoring='roc_auc', n_repeats=10,
                              random_state=0, n_jobs=-1)
imp = pd.DataFrame({'MDI': probe.feature_importances_, 'permutation': perm.importances_mean,
                    '± std': perm.importances_std}, index=Xn_train.columns)
imp['MDI rank'] = imp['MDI'].rank(ascending=False).astype(int)
imp['perm rank'] = imp['permutation'].rank(ascending=False).astype(int)
print(imp.sort_values('permutation', ascending=False).head(10).round(4))
print(imp.loc[['random_id']].round(4))

**Simulação: a importância de um par de gêmeas.** `word_freq_857` e `word_freq_415` quase sempre aparecem juntas nestes e-mails. No `probe` do passo 11, meça a queda de ROC AUC no teste ao embaralhar cada uma sozinha e as duas juntas (com a mesma permutação de linhas), dez vezes cada. A queda conjunta é maior que a soma das individuais?

In [ ]:
# sua simulação aqui

**Para fechar a aula.** Em uma frase cada: (1) por que o vencedor da grade do passo 8 não é melhor que os padrões de forma convincente? (2) Por que as probabilidades out-of-bag servem para escolher o limiar sem gastar o teste? (3) Por que `george` e `hp` são importantes neste modelo e inúteis num filtro para outra pessoa?

## Sua vez (avançado)

Para casa, com mais tempo:

1. **Pesos de classe contra limiares.** Acrescente `'class_weight': [None, 'balanced_subsample']` à grade do passo 8. A ROC AUC do vencedor muda? Reajuste-o com cada opção e compare o limiar OOB que o passo 9 escolhe. Por que pesos de classe e limiares podem acabar fazendo o mesmo trabalho ([aula 16](https://insper.github.io/ML/pt/2026.2/classes/roc-imbalanced/))?
2. **Sem os rastros do HP Labs.** Retire de `X` as colunas `word_freq_hp`, `word_freq_hpl`, `word_freq_george`, `word_freq_labs`, `word_freq_lab` e `word_freq_650` e refaça os passos 8 a 10. Quanto a ROC AUC de teste cai? Esse modelo é pior ou só mais honesto?
3. **`min_samples_leaf` pelo OOB.** Com `max_features` do vencedor, varra `min_samples_leaf` em 1, 2, 5, 10 e 20 lendo só a ROC AUC out-of-bag, como na simulação do passo 8. Onde está o melhor valor e o que acontece com o tamanho do modelo (`sum(t.tree_.node_count for t in f.estimators_)`)?